# 🔃📈📊 <span style="color: white; background-color: CadetBlue"><b> Extração do Relatório de Alteração Funcional </b></span></p>

🧩 <span style="color: Aquamarine"><b> 1- Configuração inicial e registro do processo </b></span></p>
- O script começa importando bibliotecas (Selenium, PyAutoGUI, Pandas, OpenPyXL, logging etc.)  
- Definindo caminhos (Downloads, Arquivos Movidos, Base Tratada, Processos)  
- Criando o marcador inicial tempo_0  
- Registrando Etapa 0 no arquivo de auditoria PROCESSOS.xlsx
- Isso estabelece governança desde o início da execução

🌐 <span style="color: Aquamarine"><b> 2- Acesso automático ao Datamace </b></span></p>
- A automação verifica se a janela HTML5 já está aberta (aplicação Datamace)  
- Caso afirmativo: ativa a janela e executa o login interno  
- Caso negativo:  Abre navegador via Selenium  
- Acessa https://app.dtm.com.br/  
- Realiza login Cloud  
- Aguarda carregamento  
- Executa login HTML5 via PyAutoGUI
- Esse fluxo assegura acesso robusto e consistente ao sistema

🧭 <span style="color: Aquamarine"><b> 3- Navegação até o relatório de Alteração Funcional </b></span></p>
- Após o login, o script percorre automaticamente o caminho CS → Cargos e Salários → Relatórios → Última Alteração de Cargo
- Então executa
- Configuração dos filtros necessários  
- Seleção da opção de exportação em Excel  
- Confirmação da geração do arquivo
- O arquivo é salvo no diretório de Downloads

📥 <span style="color: Aquamarine"><b> 4- Confirmação e validação do download </b></span></p>
- A automação exibe um prompt “O download da base ALTERAÇÕES foi realizado?”
- Somente após o usuário confirmar, o pipeline prossegue para o processamento

🧹 <span style="color: Aquamarine"><b> 5- Tratamento completo da base de Alteração Funcional </b></span></p>
- Para cada arquivo encontrado com prefixo REL-ULT-ALT-CARGO
- o script realiza mapeamento de colunas
- Realiza um de → para para padronização de nome das colunas
- Conversão de datas
- Campos anteriores e atuais são convertidos para datetime com tratamento de erros
- Padronização e limpeza
- Normalização de texto  
- Conversão de tipos  
- Remoção de valores inválidos  
- Preparação para conversão Excel
- Padronização da ordem das colunas
- Seguindo o modelo interno usado no RH

📦 <span style="color: Aquamarine"><b> 6- Consolidação de múltiplos arquivos </b></span></p>
- O pipeline lê todos os relatórios encontrados  
- Processa um por um  
- Acumula DataFrames válidos  
- Remove duplicidades  
- Gera uma única base consolidada
- Após essa etapa, registra Etapa 1 no PROCESSOS.xlsx

📊 <span style="color: Aquamarine"><b> 7- Geração do Excel final estruturado </b></span></p>
- Cria o arquivo ALTERAÇÕES DE FUNÇÕES.xlsx
- Com aba única  
- Tabela Excel formatada (TableStyleLight13)  
- Estrutura padronizada  
- Base limpa e pronta para dashboards, auditorias e análises internas

📁 <span style="color: Aquamarine"><b> 8- Arquivamento dos arquivos processados </b></span></p>
- Todos os arquivos originais são movidos para 2. ARQUIVOS MOVIDOS
- Mantendo histórico organizado e evitando reprocessamento indevido

🗃️ <span style="color: Aquamarine"><b> 9- Exportação para banco de dados </b></span></p>
- O script gera automaticamente tb_alteracao_funcional.csv
- Preparado para ingestão em SQL, Data Lakes, Pipelines ETL, Dashboards analíticos, etc

🔄 <span style="color: Aquamarine"><b> 10- Atualização automática da base corporativa de Alterações Funcionais </b></span></p>
-  automação abre o arquivo Base_Alteração_Funcional.xlsx
- E executa Navegação para abas relevantes (HC, Base, Consolidado) via Ctrl + G  
- Refresh de cada aba via Alt + F5
- Garantindo que todas as conexões e tabelas dinâmicas sejam atualizadas automaticamente

🧾 <span style="color: Aquamarine"><b> 11- Resumo final e encerramento </b></span></p>
- O script calcula tempo total (tempo_1 - tempo_0)  
- Registra Etapa 2 no PROCESSOS.xlsx  
- Exibe mensagem de conclusão

# Importando as Bibliotecas

In [1]:
import os
import logging
import pandas as pd
import pyautogui
import pygetwindow as gw
import pyperclip
import shutil
import socket
import sys
import time
import tkinter as tk
import win32com.client
import win32gui, win32con
from asyncio.log import logger
from contextlib import contextmanager
from datetime import datetime, date
from dotenv import load_dotenv
from openpyxl import load_workbook, Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.worksheet.table import Table, TableStyleInfo
from pathlib import Path
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import WebDriverException
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver import ActionChains
from tkinter import simpledialog
from webdriver_manager.chrome import ChromeDriverManager

tempo_0 = [id, datetime.today(), 0]

# Logging (apenas console)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Processo de Importação finalizado')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Processo de Importação finalizado

----------------------------------------------------------------------------------------------------


# Caminhos de Pastas

In [2]:
CONFIG = {
    'id_processo': 20,
    'processos': Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx'),
    'origem': Path(r'C:\Users\rodrigo.bernandes\Downloads'),
    'movidos': Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\2. ARQUIVOS MOVIDOS'),
    'saida': Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\ALTERAÇÕES DE FUNÇÕES.xlsx'),
    'env_path': Path(r'X:\Gestão de Pessoas\Analytics\08 - Notebooks Python\08.3 - Automações\.env'),
    'padrao': 'REL-ULT-ALT-CARGO',
}

# Registra etapa do processo

In [3]:
def append_registro_processo(caminho, id_proc, etapa):
    try:
        wb = load_workbook(caminho)
        ws = wb['REGISTROS']
        ws.append([id_proc, datetime.today(), etapa])
        wb.save(caminho)
        wb.close()
    except Exception as e:
        print(f"❌ Erro ao registrar etapa {etapa}: {e}")

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Registro de processos')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Registro de processos

----------------------------------------------------------------------------------------------------


# Configurações Iniciais e Acessando o Datamace

In [4]:
load_dotenv(dotenv_path='env_path')

cloud_user = os.getenv("CLOUD_USER")
cloud_password = os.getenv("CLOUD_PASSWORD")
app_user = os.getenv("APP_USER")
app_password = os.getenv("APP_PASSWORD")
datamace = os.getenv("SITE_DATAMACE")

# Validar se todas existem
if not all([cloud_user, cloud_password, app_user, app_password]):
    raise ValueError("Uma ou mais variáveis de ambiente não foram encontradas. Verifique o arquivo .env.")

titulo_aba = "HTML5"

# Verifica se janela 'HTML5' está aberta
if gw.getWindowsWithTitle(titulo_aba):
    pyautogui.click(836, 445)
    time.sleep(0.5)
    pyautogui.typewrite(app_user)
    pyautogui.press('tab')
    time.sleep(0.5)
    pyautogui.typewrite(app_password)
    pyautogui.press('enter')
else:
    driver = webdriver.Chrome()
    time.sleep(1)
    window = win32gui.GetForegroundWindow()
    win32gui.ShowWindow(window, win32con.SW_MAXIMIZE)
    time.sleep(1)
    driver.get(datamace)
    time.sleep(8)
    # Login cloud
    driver.find_element(By.NAME, "username").send_keys(cloud_user)
    driver.find_element(By.NAME, "Password").send_keys(cloud_password)
    driver.find_element(By.NAME, "Password").send_keys(Keys.ENTER)
    time.sleep(30)
    # Login pyautogui
    pyautogui.click(836, 445)
    time.sleep(0.5)
    pyautogui.typewrite(app_user)
    pyautogui.press('tab')
    time.sleep(0.5)
    pyautogui.typewrite(app_password)
    pyautogui.press('enter')

print("🏁 Acesso finalizado")

🏁 Acesso finalizado


# Acessando o CS

In [5]:
# Fechando janelas de Novidades do Datamace
time.sleep(3)
pyautogui.press('enter')
time.sleep(3)
pyautogui.click(x=1268, y=199)
# Acessando o CS
time.sleep(10) # Tempo maior para aguardar carregar o CS
pyautogui.click(x=268, y=193) # Clicando no módulo CS
time.sleep(3)
pyautogui.click(x=1075, y=239) # Fecha a janela de Tarefas Anuais
time.sleep(2)
pyautogui.click(x=451, y=298) # Cargos e Salários
time.sleep(2)
pyautogui.click(x=523, y=451) # Relatórios
time.sleep(2)
pyautogui.click(x=713, y=846) # Última Alteração de Cargo
time.sleep(2)
pyautogui.click(x=664, y=414) # Multi processamento
time.sleep(2)
pyautogui.click(x=848, y=478) # Imprime dados do cargo anterior?
time.sleep(2)
pyautogui.click(x=845, y=508) # Sim
time.sleep(2)
pyautogui.click(x=752, y=533) # Situação
time.sleep(2)
pyautogui.click(x=724, y=572) # Todos
time.sleep(2)
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(2)
pyautogui.click(x=547, y=704) # Confirmar
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Extraindo relatório')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Extraindo relatório

----------------------------------------------------------------------------------------------------


# Aguardando a Conclusão do Download da Base ALTERACOES

In [6]:
# Configuração de logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

def verificar_download_base() -> bool:
    """
    Exibe uma caixa de diálogo para confirmar se o download da base COLAB foi realizado.
    """
    tipo_escolhido = pyautogui.confirm(
        text='Foi realizado o download da base de alterações funcionais?', 
        title='Seleção de Extração', 
        buttons=['Sim']
    )

    # Se fechar a janela
    if tipo_escolhido is None:
        logging.warning("Operação cancelada pelo usuário no prompt inicial. ❌ Encerrando o script.")
        sys.exit(0)
        
    # Se for 'Sim'
    logging.info(f"Opção selecionada: {tipo_escolhido}. ✅ Continuando o processo...")
    return True

# --- Execução principal ---
if __name__ == "__main__":
    # Chama a função de validação antes de seguir com o restante do código
    verificar_download_base()
    
    logging.info("Iniciando a leitura e processamento dos dados...")

2026-06-24 09:50:14,847 - INFO - Opção selecionada: Sim. ✅ Continuando o processo...
2026-06-24 09:50:14,876 - INFO - Iniciando a leitura e processamento dos dados...


# Mapeamento de Colunas

In [7]:
MAPEAMENTO_COLUNAS = {
    'EMPRESA': 'cod_empresa',
    'REGISTRO': 'registro',
    'NOME': 'nome',          
    'DIRET': 'cod_diretoria',
    'DEPTO': 'depto',
    'SETOR': 'cod_setor',
    'SECAO': 'cod_secao',
    'CCUSTO': 'cod_centro_custo',
    'CARGO ANT.': 'cargo_anterior',
    'CBO ANT.': 'cod_cbo',
    'MOTIVO ANT': 'motivo_anterior', 
    'DT_REF_ANT': 'data_ref_anterior',
    'CARGO ATU.': 'cargo', 
    'CBO ATUAL': 'cbo_atual',    
    'MOTIVO ATU': 'motivo_atual',     
    'DT_REF_ATU': 'data_ref_atual'
}
COLUNAS_DATA = ['DT_REF_ANT', 'DT_REF_ATU']

# Processo de Carga

In [8]:
@contextmanager

def gerenciar_workbook(caminho: Path, sheet: str):
    """Context manager para operações seguras com workbook."""
    wb = load_workbook(caminho)
    ws = wb[sheet]
    try:
        yield ws
    finally:
        wb.save(caminho)

def registrar_etapa(caminho: Path, id_proc: int, etapa: int):
    """Registra etapa do processo."""
    with gerenciar_workbook(caminho, 'REGISTROS') as ws:
        ws.append([id_proc, datetime.today(), etapa])
    logger.debug(f"✅ Etapa {etapa} registrada")

def converter_datas(df: pd.DataFrame, colunas: list) -> pd.DataFrame:
    """Converte colunas para datetime."""
    for col in colunas:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], format='%d/%m/%Y', errors='coerce')
    return df

def mapear_colunas_acidentes(df: pd.DataFrame) -> pd.DataFrame:
    """Mapeia colunas do arquivo de acidentes."""
    colunas_existentes = {k: v for k, v in MAPEAMENTO_COLUNAS.items() if k in df.columns}
    return df.rename(columns=colunas_existentes)[list(colunas_existentes.values())]

def processar_arquivo(caminho: Path) -> pd.DataFrame:
    """Processa um arquivo de acidentes."""
    try:
        logger.debug(f"Carregando: {caminho.name}")
        
        # Carregar
        df = pd.read_excel(caminho, dtype={'REGISTRO': str}, engine='openpyxl')
        
        # Limpar
        df = df.dropna(subset=['NOME'])
        
        # Converter tipos
        #df['REGISTRO'] = pd.to_numeric(df['REGISTRO'], errors='coerce')
        df = converter_datas(df, COLUNAS_DATA)
        
        # Mapear colunas
        df = mapear_colunas_acidentes(df)
        
        logger.debug(f"✅ {len(df)} registros processados")
        return df
        
    except Exception as e:
        logger.error(f"❌ Erro ao processar {caminho.name}: {e}")
        return None
    
def criar_excel_com_tabela(df: pd.DataFrame, caminho: Path):
    """Cria Excel com tabela formatada."""
    logger.info(f"📝 Criando Excel final ({len(df)} registros)...")
    
    # Salvar com Pandas
    df.to_excel(caminho, sheet_name='ALTERACOES', index=False, engine='openpyxl')
    
    # Aplicar formatação
    wb = load_workbook(caminho)
    ws = wb.active
    
    tabela = Table(displayName="ALTERACOES", ref=ws.dimensions)
    estilo = TableStyleInfo(
        name="TableStyleLight13",
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=True,
        showColumnStripes=True
    )
    tabela.tableStyleInfo = estilo
    ws.add_table(tabela)
    
    wb.save(caminho)
    logger.info(f"✅ Excel criado: {caminho}")
    
# Main
if __name__ == "__main__":
    tempo_inicio = datetime.now()
    
    logger.info("=" * 80)
    logger.info("🔄 PROCESSAMENTO DE ALTERAÇÕES DE FUNÇÕES")
    logger.info("=" * 80)

2026-06-24 09:50:14,990 - INFO - ================================================================================
2026-06-24 09:50:14,992 - INFO - 🔄 PROCESSAMENTO DE ALTERAÇÕES DE FUNÇÕES
2026-06-24 09:50:14,993 - INFO - ================================================================================


# Processamento e Consolidação do Arquivo

In [9]:
try:
    # Etapa 1: Registrar início
    registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 0)
        
    # Etapa 2: Buscar arquivos
    logger.info("\n📂 Buscando arquivos...")
    arquivos = [f for f in CONFIG['origem'].iterdir() if f.name.startswith(CONFIG['padrao'])]
        
    if not arquivos:
        logger.warning("❌ Nenhum arquivo encontrado")
        exit()
        
    logger.info(f"✅ {len(arquivos)} arquivo(s) encontrado(s)\n")
        
    registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 1)
        
    # Etapa 3: Processar arquivos
    logger.info("🔄 Processando arquivos...\n")
        
    todas_bases = []
        
    for idx, arquivo in enumerate(arquivos, 1):
        logger.info(f"[{idx}/{len(arquivos)}] {arquivo.name}")
            
        df = processar_arquivo(arquivo)
            
        if df is not None and len(df) > 0:
            todas_bases.append(df)
                
            try:
                destino = CONFIG['movidos'] / arquivo.name
                shutil.move(str(arquivo), str(destino))
                logger.info(f"✅ Movido para arquivos processados\n")
            except Exception as e:
                logger.error(f"⚠️ Erro ao mover: {e}\n")
        else:
            logger.warning(f"❌ Sem dados válidos\n")
        
    registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 2)
        
    # Etapa 4: Consolidar e salvar
    if todas_bases:
        logger.info("💾 Consolidando dados...")
        base_final = pd.concat(todas_bases, ignore_index=True)
        base_final = base_final.drop_duplicates()
        logger.info(f"✅ {len(base_final)} registros consolidados\n")
            
        criar_excel_com_tabela(base_final, CONFIG['saida'])
    else:
        logger.error("❌ Nenhum arquivo foi processado!")
        exit()
        
    # Finalizar
    tempo_duracao = datetime.now() - tempo_inicio
        
    logger.info("\n" + "=" * 80)
    logger.info("✅ PROCESSO FINALIZADO COM SUCESSO!")
    logger.info(f"   Tempo de execução: {tempo_duracao}")
    logger.info("=" * 80)
        
except Exception as e:
    logger.error(f"\n❌ ERRO CRÍTICO: {e}")
    import traceback
    logger.error(traceback.format_exc())

2026-06-24 09:50:15,450 - INFO - 
📂 Buscando arquivos...
2026-06-24 09:50:15,451 - INFO - ✅ 1 arquivo(s) encontrado(s)

2026-06-24 09:50:15,763 - INFO - 🔄 Processando arquivos...

2026-06-24 09:50:15,764 - INFO - [1/1] REL-ULT-ALT-CARGO-09333040.XLSX
c:\Users\rodrigo.bernandes\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
2026-06-24 09:50:18,003 - INFO - ✅ Movido para arquivos processados

2026-06-24 09:50:18,369 - INFO - 💾 Consolidando dados...
2026-06-24 09:50:18,384 - INFO - ✅ 10701 registros consolidados

2026-06-24 09:50:18,384 - INFO - 📝 Criando Excel final (10701 registros)...
2026-06-24 09:50:25,230 - INFO - ✅ Excel criado: X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\ALTERAÇÕES DE FUNÇÕES.xlsx
2026-06-24 09:50:25,231 - INFO - 
2026-06-24 09:50:25,231 - INFO - ✅ PROCESSO FIN

# Criando Base em CSV para o Banco de Dados

In [10]:
# Ler o XLSX
df = pd.read_excel(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\ALTERAÇÕES DE FUNÇÕES.xlsx')

# Salvar como CSV
df.to_csv(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\tb_alteracoes_funcoes.csv', index=False, encoding='utf-8')

print("✅ Arquivo convertido para CSV com sucesso!")

✅ Arquivo convertido para CSV com sucesso!


# Atualizando o Arquivo Alteração Funcional

In [11]:
# Caminho do arquivo
path_excel = r"X:\Gestão de Pessoas\Analytics\10 - Relatórios\10.6 - Alteração Funcional\Base_Alteração_Funcional.xlsx"
os.startfile(path_excel) # Abre o arquivo
time.sleep(10)

# Atualizando aba HC
pyautogui.hotkey('ctrl', 'g')
time.sleep(1)
pyautogui.write('HC!B5')
time.sleep(1)
pyautogui.press('enter')
time.sleep(2)
pyautogui.hotkey('alt', 'f5')
time.sleep(10)

# Atualizando aba Base Alterações
pyautogui.hotkey('ctrl', 'g')
time.sleep(1)
pyautogui.write('Base!B5')
time.sleep(1)
pyautogui.press('enter')
time.sleep(2)
pyautogui.hotkey('alt', 'f5')
time.sleep(10)

# Atualizando aba Alterações Funcionais
pyautogui.hotkey('ctrl', 'g')
time.sleep(1)
pyautogui.write('Consolidado!B5')
time.sleep(1)
pyautogui.press('enter')
time.sleep(2)
pyautogui.hotkey('alt', 'f5')

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Planilha atualizada com sucesso')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Planilha atualizada com sucesso

----------------------------------------------------------------------------------------------------


# Resumo de Finalização do Processo

In [12]:
tempo_1 = [id, datetime.today(), 1]

print('----------------------------------------------------------------------------------------------------')
print('')
print('     ✅  Processo finalizado')
print('')
print('     ⏱️   Tempo de execução:')
print('')
print(f'   {tempo_1[1] - tempo_0[1]}')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

     ✅  Processo finalizado

     ⏱️   Tempo de execução:

   0:18:23.107418

----------------------------------------------------------------------------------------------------
